In [5]:
import pandas as pd

# 1. Đọc file dữ liệu khuyến mãi từ định dạng JSON (Hỗ trợ cả JSON chuẩn và JSON Lines)
json_promo_file = 'eprom.json'  # Nếu tên file là promotions_2.json thì bạn đổi tên ở đây nhé

try:
    df_promo = pd.read_json(json_promo_file)
except ValueError:
    df_promo = pd.read_json(json_promo_file, lines=True)

print(f'Số dòng ban đầu (promotion): {len(df_promo)}')

# 2. Xóa các dòng bị trùng lặp hoàn toàn
df_promo = df_promo.drop_duplicates()

# 3. Làm sạch chuỗi văn bản (strip khoảng trắng thừa)
text_cols_promo = df_promo.select_dtypes(include=['object']).columns
for col in text_cols_promo:
    df_promo[col] = df_promo[col].astype(str).str.strip()

# 4. Chuẩn hóa tên cột về dạng snake_case (Làm trước để dễ tham chiếu bên dưới)
df_promo.columns = (
    df_promo.columns.str.strip().str.lower().str.replace(' ', '_')
)

# 5. Xử lý giá trị khuyết thiếu (Missing values) - Bắt buộc phải có promo_id
if 'promo_id' in df_promo.columns:
    df_promo = df_promo.dropna(subset=['promo_id'])

# 6. Chuyển đổi kiểu dữ liệu cho đúng chuẩn (float cho discount, datetime cho ngày tháng)
if 'discount_value' in df_promo.columns:
    df_promo['discount_value'] = pd.to_numeric(
        df_promo['discount_value'], errors='coerce'
    ).fillna(0.0).astype(float)

if 'start_date' in df_promo.columns:
    df_promo['start_date'] = pd.to_datetime(df_promo['start_date'], errors='coerce')

if 'end_date' in df_promo.columns:
    df_promo['end_date'] = pd.to_datetime(df_promo['end_date'], errors='coerce')

# 7. Xuất ra file promotion đã làm sạch chuẩn
output_filename = 'promotion.csv'
df_promo.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f'✅ Đã làm sạch và tạo xong file: {output_filename} (còn {len(df_promo)} dòng)')
print(df_promo.head())

Số dòng ban đầu (promotion): 1000
✅ Đã làm sạch và tạo xong file: promotion.csv (còn 1000 dòng)
          promo_id          promo_name  promo_type  discount_value start_date  \
0  PROMO-0039-0001    Fall Launch 2020  percentage            10.0 2020-08-30   
1  PROMO-0029-0002    Fall Launch 2018  percentage            10.0 2018-08-30   
2  PROMO-0015-0003  Urban Blowout 2015       fixed            50.0 2015-07-30   
3  PROMO-0043-0004    Fall Launch 2021  percentage            10.0 2021-08-30   
4  PROMO-0008-0005  Mid-Year Sale 2014  percentage            18.0 2014-06-23   

    end_date applicable_category promo_channel  stackable_flag  \
0 2020-10-01                None  all_channels               0   
1 2018-10-01                None         email               0   
2 2015-09-02          Streetwear        online               0   
3 2021-10-02                None         email               0   
4 2014-07-22                None  social_media               0   

   min_order_value  